In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

from sklearn.linear_model import LinearRegression,Ridge,ElasticNet,Lasso
from sklearn.metrics import mean_squared_error,mean_absolute_error,accuracy_score,r2_score

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.decomposition import PCA

from statsmodels.tsa.stattools import adfuller
from pylab import rcParams
import statsmodels.api as sm
from statsmodels.tsa.arima_model import ARIMA

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout
from keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import GridSearchCV
import warnings 
warnings.filterwarnings('ignore')

In [ ]:
df_train=pd.read_csv('train_data.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.info()

In [ ]:
df_test=pd.read_csv('test_data.csv')

In [ ]:
df_test.head()

In [ ]:
print('Train_data:',df_train.shape,df_train.size,'\n','Test_data:',
      df_test.shape,df_test.size)

In [ ]:
df_test.info()

In [ ]:
df_train.isna().sum()

In [ ]:
df_test.isna().sum()

In [ ]:
df_train['StateHoliday'].unique()

In [ ]:
sh={'0':0, 'a':1, 'b':2, 'c':3,0:0}


In [ ]:
df_train['StateHoliday']=df_train['StateHoliday'].map(sh)


In [ ]:
df_test['StateHoliday']=df_test['StateHoliday'].map(sh)

In [ ]:
df_test['StateHoliday'].unique()

In [ ]:
df_train.describe()

In [ ]:
df_train.DayOfWeek.value_counts()

In [ ]:
df_test.DayOfWeek.value_counts()

In [ ]:
df_train.Open.value_counts()

In [ ]:
df_test.Open.value_counts()

In [ ]:
df_train.Promo.value_counts()

In [ ]:
df_test.Promo.value_counts()

In [ ]:
df_train.StateHoliday.value_counts()

In [ ]:
df_test.StateHoliday.value_counts()

In [ ]:
df_train.SchoolHoliday.value_counts()

In [ ]:
df_test.SchoolHoliday.value_counts()

In [ ]:
df_train['Date']=pd.to_datetime(df_train['Date'], format='%Y-%m-%d')
df_test['Date']=pd.to_datetime(df_test['Date'], format='%d-%m-%Y')

In [ ]:
#feature engineering
import datetime
df_train['day']=pd.to_datetime(df_train['Date'], format='%Y-%m-%d').dt.day
df_train['month']=pd.to_datetime(df_train['Date'], format='%Y-%m-%d').dt.month
df_train['year']=pd.to_datetime(df_train['Date'], format='%Y-%m-%d').dt.year

In [ ]:
df_test['day']=pd.to_datetime(df_test['Date'], format='%d-%m-%Y').dt.day
df_test['month']=pd.to_datetime(df_test['Date'], format='%d-%m-%Y').dt.month
df_test['year']=pd.to_datetime(df_test['Date'], format='%d-%m-%Y').dt.year

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
# distribution plot
sns.distplot(df_train['Sales'])

In [ ]:
sns.distplot(df_train['Customers'])

In [ ]:
# boxplots
sns.barplot(x=df_train['DayOfWeek'],y=df_train['Sales'])

In [ ]:
sns.barplot(x=df_train['SchoolHoliday'],y=df_train['Sales'])

In [ ]:
df_train.columns

In [ ]:
df_train['Date'].dtype

In [ ]:
df_test['Date'].dtype

#### Building Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
import numpy as np

# Split data into training and test sets
X=df_train.drop(['Sales','Date'],axis=1)
y=df_train['Sales']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42)

# Fit the linear regression model
reg = LinearRegression().fit(X_train, y_train)

# Make predictions on the test set
y_pred = reg.predict(X_test)

# Evaluate the model using RMSE and MAE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
print(f'RMSE: {rmse}, MAE: {mae}')

score_lr=cross_val_score(reg,X,y,cv=5)
print(score_lr)


In [ ]:

# Create a dictionary to store the models
models = {}

# Loop over the unique store IDs
for store_id in df_train['Store'].unique():
    # Filter the data to only include the current store
    store_data = df_train[df_train['Store'] == store_id]
    
    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(store_data.drop(['Sales','Date'], axis=1), store_data['Sales'], test_size=0.2)
    
    # Fit the linear regression model
    models[store_id] = LinearRegression().fit(X_train, y_train)
    
    # Make predictions on the test set
    y_pred = models[store_id].predict(X_test)
    
    # Print the performance metrics
    print(f'Store ID: {store_id}')
    print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred))}, MAE: {mean_absolute_error(y_test, y_pred)}')


In [ ]:
df_train['Store'].unique().sum()

#### Use Non-Linear Regressors like Random Forest or other Tree-based Regressors:

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
dt=DecisionTreeRegressor()
dt.fit(X_train,y_train)
pred_dt=dt.predict(X_test)
score_dt=cross_val_score(dt,X,y,cv=5)
print(score_dt)

In [ ]:
score_dt.mean()

In [ ]:
mae_dt=mean_absolute_error(y_test,pred_dt)
print(mae_dt)
mse_dt=mean_squared_error(y_test,pred_dt)
print(mae_dt)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor()
rf.fit(X_train,y_train)
pred_rf=rf.predict(X_test)
score_rf=cross_val_score(rf,X,y,cv=5)
print(score_rf)

In [ ]:
score_rf.mean()

In [ ]:
mae_rf=mean_absolute_error(y_test,pred_rf)
print(mae_rf)
mse_rf=mean_squared_error(y_test,pred_rf)
print(mae_rf)

In [ ]:
combi1= pd.get_dummies(df_train,columns=['DayOfWeek', 'Open', 'Promo','StateHoliday', 'SchoolHoliday', 'year','Store','day','month'],drop_first=True)
# combi1=pd.read_csv('combi1.csv')
# combi1.drop(['Unnamed: 0'],axis=True,inplace=True)
combi1.head()

In [ ]:
df_train.corr()['Sales']

#### Dimensional Reduction techniques like, PCA and Tree’s Hyperparameter Tuning will be required. Cross-validate to find the best parameters. Infer the performance of both the models. 

In [ ]:
X_train = df_train.drop(['Sales','Date','Open','Customers'],axis=1).values

pca = PCA().fit(X_train)
#Plotting the Cumulative Summation of the Explained Variance
plt.figure()
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Variance (%)') #for each component
plt.title('Explained Variance')
plt.show()




In [ ]:
X = df_train.drop(['Sales','Date','Open','Customers'],axis=1).values
#X_val = test_val1.drop(['Sales','Date','Open','Customers'],axis=1).values
y = df_train['Sales']
#Y_val = test_val1['Sales']

pca = PCA(n_components=2)  
X_train = pca.fit_transform(X)  
#X_val= pca.transform(X_val) 

clf = RandomForestRegressor(n_estimators=50,max_features='sqrt',max_depth=6,random_state=0,n_jobs=7)
clf.fit(X,y)
pred1 = clf.predict(X)

#ind=test_val[test_val.Open==0].index
#for i in ind:
    #pred1[i] = 0
    
print('MSE',np.sqrt(mean_squared_error(pred1,y)))
print('MAE',mean_absolute_error(pred1,y))

#### Train a Time-series model on the data taking time as the only feature. This will be a store-level training.

Identify yearly trends and seasonal months

In [ ]:
dateparse = lambda dates: pd.datetime.strptime(dates, '%Y-%m-%d')
Train = pd.read_csv("train_data.csv", parse_dates=['Date'], index_col='Date',date_parser=dateparse)
Test_val = pd.read_csv("test_data_hidden.csv", parse_dates=['Date'], index_col='Date',date_parser=dateparse)
Train=Train[['Store','Sales','Open','DayOfWeek']]
Test_val=Test_val[['Store','Sales','Open','DayOfWeek']]
print ('\n Parsed Data:')
Train.sort_values(['Date'],axis=0,inplace=True)
Test_val.sort_values(['Date'],axis=0,inplace=True)
print (Train.head())

In [ ]:
store1=Train[Train.Store==1]
test_store1=Test_val[Test_val.Store==1]

In [ ]:
sns.boxplot(x="DayOfWeek", y="Sales", data=store1)

Monday=1, Sunday=7.

Here we can find on Sunday stores are closed. Monday has little larger sales, Thurdays has little smaller. There's a few outliers on all days(except Sunday) but it is less on Weekdays(1,3)

In [ ]:
from statsmodels.tsa.stattools import adfuller
from pylab import rcParams
import statsmodels.api as sm
from statsmodels.tsa.arima_model import ARIMA

rcParams['figure.figsize'] = 18, 8
decomposition = sm.tsa.seasonal_decompose(store1['Sales'], model='additive',freq=365)
fig = decomposition.plot()
plt.show()

In [ ]:
def test_stationarity(timeseries):
    
    #Determing rolling statistics
    rolmean = timeseries.rolling(12).mean()
    rolstd = timeseries.rolling(12).std()
    
    #Plot rolling statistics:
    orig = plt.plot(timeseries, color='blue',label='Original')
    mean = plt.plot(rolmean, color='red', label='Rolling Mean')
    std = plt.plot(rolstd, color='black', label = 'Rolling Std')
    
    plt.legend(loc='best')
    plt.title('Rolling Mean & Standard Deviation')
    plt.show(block=False)
    
    #Perform Dickey-Fuller test:
    print('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries, autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)
    

In [ ]:
test_stationarity(store1['Sales'])

The smaller p-value, the more likely it's stationary. Here our p-value is 0.000415. It's actually good, but as we just visually found a little downward trend, we want to be more strict, i.e. if the p value further decreases, this series would be more likely to be stationary. To get a stationary data, there's many techiniques. We can use log, differencing etc..

In [ ]:
first_diff = store1['Sales'] - store1['Sales'].shift(1)
first_diff = first_diff.dropna(inplace = False)
test_stationarity(first_diff)

After differencing, the p-value is extremely small. Thus this series is very likely to be stationary.

In [ ]:
#AR model
ar_mod = ARIMA(store1.Sales, (9,1,0),freq='D')
res=ar_mod.fit(disp=False)
Y_pred = res.forecast(steps=31)[0]
print('MSE',np.sqrt(mean_squared_error(Y_pred,test_store1.Sales)))
print('MAE',mean_absolute_error(Y_pred,test_store1.Sales))

In [ ]:
#MA model
ma_mod = ARIMA(store1.Sales, (0,1,1),freq='D')
res=ma_mod.fit(disp=False)
Y_pred = res.forecast(steps=31)[0]
print('MSE',np.sqrt(mean_squared_error(Y_pred,test_store1.Sales)))
print('MAE',mean_absolute_error(Y_pred,test_store1.Sales))

In [ ]:
#ARIMA model
arima_mod = ARIMA(store1.Sales, (9,1,9),freq='D')
res=arima_mod.fit(disp=False)
Y_pred = res.forecast(steps=31)[0]
print('MSE',np.sqrt(mean_squared_error(Y_pred,test_store1.Sales)))
print('MAE',mean_absolute_error(Y_pred,test_store1.Sales))


#### Project Task: Week 2

Implementing Neural Networks:

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout
from keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
train_store1 = store1.iloc[:, 1:2].values
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler(feature_range = (0, 1))
train_store1 = sc.fit_transform(train_store1)

X_train = []
Y_train = []
for i in range(30, 911):
    X_train.append(train_store1[i-30:i, 0])
    Y_train.append(train_store1[i, 0])
X_train, Y_train = np.array(X_train), np.array(Y_train)
# Reshaping
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [ ]:
regressor = Sequential()
regressor.add(LSTM(units = 30, return_sequences = True, input_shape = (X_train.shape[1], 1)))
regressor.add(LSTM(units = 50, return_sequences = True))
regressor.add(LSTM(units = 70, return_sequences = True))
regressor.add(LSTM(units = 50))
regressor.add(Dense(units = 1))
regressor.compile(optimizer = 'adam', loss = 'mean_squared_error')
regressor.fit(X_train, Y_train, epochs = 100, batch_size = 64, shuffle=False)


In [ ]:
total_data = pd.concat((store1['Sales'], test_store1['Sales']), axis = 0)
inputs = total_data[len(total_data) - len(test_store1) - 30:].values
inputs = inputs.reshape(-1,1)
inputs = sc.transform(inputs)
X_test = []
for i in range(30, 61):
    X_test.append(inputs[i-30:i, 0])
X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
pred = regressor.predict(X_test)
pred= sc.inverse_transform(pred)
print(np.sqrt(mean_squared_error(pred,test_store1.Sales)))
print(mean_absolute_error(pred,test_store1.Sales)) 

In [ ]:
# Visualising the results
plt.plot(test_store1.Sales, color = 'red', label = 'Actual Sales')
plt.plot(pred, color = 'blue', label = 'Predicted Sales')
plt.title('Sales Prediction')
plt.xlabel('Time')
plt.ylabel('Sale')
plt.legend()
plt.show()

#### Applying ANN:

In [ ]:
#Model1
X_train = df_train.drop(['Sales','Date','Customers'],axis=1).values
#X_val = test_val2.drop(['Sales','Date','Customers'],axis=1).values
Y_train = pd.DataFrame(df_train['Sales'])
#Y_val = test_val2['Sales']
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler(feature_range = (0, 1))
Y_train = sc.fit_transform(Y_train)

model = Sequential()
model.add(Dense(100, activation='relu', input_dim = X_train.shape[1]))
#model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dense(50, activation='relu'))
#model.add(Dropout(0.2))
model.add(Dense(1,activation='linear',kernel_initializer='normal') )
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, Y_train, epochs=10, batch_size=64,shuffle=False,verbose=0)
Y_pred = model.predict(X_train, batch_size=64,verbose=0)
Y_pred= sc.inverse_transform(Y_pred)
print('MSE',np.sqrt(mean_squared_error(Y_pred,Y_train)))
print('MAE',mean_absolute_error(Y_pred,Y_train))


In [ ]:
#model2
X_train = df_train.drop(['Sales','Date','Customers'],axis=1).values
#X_val = test_val1.drop(['Sales','Date','Customers'],axis=1).values
Y_train = pd.DataFrame(df_train['Sales'])
#Y_val = test_val1['Sales']
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler(feature_range = (0, 1))
Y_train = sc.fit_transform(Y_train)

model = Sequential()
model.add(Dense(100, activation='relu', input_dim = X_train.shape[1]))
#model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1,activation='linear') )
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, Y_train, epochs=10, batch_size=64,shuffle=False,verbose=0)
Y_pred = model.predict(X_train, batch_size=64,verbose=0)
Y_pred= sc.inverse_transform(Y_pred)


print('MSE',np.sqrt(mean_squared_error(Y_pred,Y_train)))
print('MAE',mean_absolute_error(Y_pred,Y_train))